# Лабораторная работа №1  
## Разведочный анализ данных. Исследование и визуализация

**Датасет:** Wine recognition (скачивается через `sklearn.datasets`, без пропусков).

In [ ]:
from sklearn.datasets import load_wine

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
%matplotlib inline

wine = load_wine(as_frame=True)
wine

## 1. Текстовое описание выбранного набора данных

**Источник:** репозиторий UCI Machine Learning Repository, встроенная копия в [scikit-learn](https://scikit-learn.org/stable/datasets/toy_dataset.html#wine-dataset).

**Предметная область:** химический анализ вин, выращенных в одном регионе Италии тремя разными производителями (классы 0, 1, 2 — условные сорта/производители).

**Признаки** (все вещественные): концентрации и соотношения различных веществ (например, алкоголь, яблочная кислота, фенолы, антоцианы, оттенок, пролин и др.). **Целевая переменная** — дискретный класс вина (3 класса).

Набор небольшой (178 объектов), **пропусков нет**, что упрощает первичную визуализацию без отдельного этапа заполнения пропусков — в соответствии с рекомендациями к лабораторной работе.

In [ ]:
X = wine.data
y = wine.target.rename("target")
df = pd.concat([X, y], axis=1)
feature_names = list(wine.feature_names)
df.head()

## 2. Основные характеристики датасета

Размерность, типы столбцов, базовые статистики, распределение целевого признака.

In [ ]:
print("Размер таблицы (строки, столбцы):", df.shape)
print("Число классов:", df["target"].nunique())
df.info()

In [ ]:
display(df.describe().T.round(4))

In [ ]:
counts = df["target"].value_counts().sort_index()
counts

## 3. Визуальное исследование датасета

Гистограммы количественных признаков (для универсальной совместимости сборок KDE отключено в `sns.histplot`; при желании можно задать `kde=True` локально), распределение по классу, фрагмент парных двумерных проекций, ящики с усами для сравнения классов по отдельным признакам.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(14, 12))
axes = axes.ravel()
for ax, col in zip(axes, feature_names):
    sns.histplot(df[col], kde=False, ax=ax)
    ax.set_title(col)
for ax in axes[len(feature_names) :]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="target")
plt.title("Число объектов по классам")
plt.xlabel("Класс вина")
plt.ylabel("Количество")
plt.show()

In [ ]:
# Парная визуализация через matplotlib (устойчивее, чем sns.pairplot в части сборок Python)
pair_cols = ["alcohol", "malic_acid", "color_intensity", "proline"]
targets = sorted(df["target"].unique())
n = len(pair_cols)
fig, axes = plt.subplots(n, n, figsize=(11, 10))
palette = sns.color_palette("tab10", len(targets))
for i, yi in enumerate(pair_cols):
    for j, xj in enumerate(pair_cols):
        ax = axes[i, j]
        if i == j:
            ax.hist(df[yi], bins=14, alpha=0.85, color="steelblue")
            ax.set_ylabel("частота" if j == 0 else "")
        elif i > j:
            for ti, cls in enumerate(targets):
                sub = df.loc[df["target"] == cls]
                ax.scatter(
                    sub[xj],
                    sub[yi],
                    alpha=0.65,
                    s=22,
                    color=palette[ti],
                    label=str(cls),
                )
            if i == n - 1:
                ax.set_xlabel(xj)
            if j == 0:
                ax.set_ylabel(yi)
            if i == n - 1 and j == n - 2:
                ax.legend(title="класс", fontsize=8, title_fontsize=8)
        else:
            ax.axis("off")

plt.suptitle("Фрагмент парных связей между признаками (подматрица)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
sample_feats = [
    "alcohol",
    "flavanoids",
    "color_intensity",
    "proline",
]
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.ravel()
for ax, col in zip(axes, sample_feats):
    sns.boxplot(data=df, x="target", y=col, ax=ax)
    ax.set_title(col)
plt.suptitle("Сравнение классов по отдельным признакам (boxplot)")
plt.tight_layout()
plt.show()

## 4. Информация о корреляции признаков

Матрица линейных корреляций Пирсона между **вещественными** признаками (без целевой метки как числовой переменной — она порядковая по смыслу; при необходимости её можно анализировать отдельно через групповые средние). Тепловая карта наглядно показывает сильные положительные и отрицательные связи.

In [ ]:
corr = X.corr(method="pearson")
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(11, 9))
sns.heatmap(
    corr,
    mask=mask,
    cmap="vlag",
    center=0,
    square=True,
    linewidths=0.3,
    cbar_kws={"shrink": 0.6},
)
plt.title("Корреляция Пирсона между признаками (нижний треугольник)")
plt.tight_layout()
plt.show()

In [ ]:
strong = (
    corr.where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
    .stack()
    .abs()
    .sort_values(ascending=False)
)
print("Топ-10 пар по модулю корреляции:")
display(strong.head(10).to_frame("|r|"))

### Краткие выводы

- Датасет **Wine** содержит 178 наблюдений и 13 числовых признаков без пропусков; задача многоклассовой классификации с тремя классами (классы слегка **несбалансированы** — это видно по countplot).
- Распределения признаков **различны по форме**: часть близка к унимодальному виду, часть более скошена — универсальную нормальность данных предполагать нельзя.
- На матрице scatter-графиков заметимо **частичное разделение классов** по комбинациям (например, `proline`, `color_intensity`, `alcohol`).
- Матрица корреляций показывает **существенные линейные связи** между отдельными химическими показателями; сильнейшие пары перечислены в последней кодовой ячейке — их полезно иметь в виду при отборе признаков и интерпретации моделей.